# TxSON Data Pipeline — Script Demo

This notebook walks through each script in `scripts/`, in the order they're used in the pipeline. Every script except `soil_or_met.py` and `read_data.py` can also be run from the command line — each section shows the CLI.

**Library modules** (imported only, no CLI)
1. `soil_or_met.py` — classify a raw `.dat` file as soil, met, or unknown
2. `read_data.py` — read a raw `.dat` file into a DatetimeIndexed DataFrame

**Prewash steps** (one station's DataFrame in, cleaned DataFrame out)

3. `dup_cleaner.py` — resolve duplicate timestamps
4. `treat_subhourly_data.py` — condense sub-hourly readings onto the hourly grid
5. `time_cleaner.py` — fill missing hourly timestamps with NaN
6. `treat_wrong_data.py` — replace out-of-range values with NA

**Diagnostics & orchestration**

7. `gap_report.py` — report data gaps by duration
8. `get_data_dict.py` — run the whole pipeline over a folder of raw files

Prewash order: **`dup_cleaner` → `treat_subhourly_data` → `time_cleaner` → `treat_wrong_data`**.

In [3]:
# Shared setup — edit this path to point to your raw data folder.
# The raw data ships in the repo one level up from scripts/, so this relative path
# works when the notebook is run from the scripts/ folder.
DATA_FOLDER = r"C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24"

import os

# Pick one example soil file and one example met file for the single-file demos.
all_files = [f for f in os.listdir(DATA_FOLDER) if f.endswith(".dat")]

# A met file has "_met" in the name.
example_met_file  = next((os.path.join(DATA_FOLDER, f) for f in all_files if "_met" in f.lower()), None)
example_soil_file = next((os.path.join(DATA_FOLDER, f) for f in all_files if "_met" not in f.lower()), None)

print(f"Met  file: {os.path.basename(example_met_file)}")
print(f"Soil file: {os.path.basename(example_soil_file)}")

Met  file: CB01_met.dat
Soil file: CB01.dat


---
## 1. `soil_or_met.py`  ·  *library module, no CLI*

`SoilOrMet.determine_data_file()` inspects the first 10 lines of a `.dat` file and classifies it as `'soil'`, `'met'`, or `'unknown'` based on:
- presence of a `YYYY-MM-DD HH:MM:SS` timestamp
- overlap with the expected soil / met column headers

In [4]:
from soil_or_met import SoilOrMet

# default config (shown explicitly for reference):
# SOIL_HEADER = "Date,Ppt,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag"
# MET_HEADER  = "TIMESTAMP,RECORD,Rain_mm_Tot,AirTC_Avg,RH,WS_ms_S_WVT,WindDir_D1_WVT,SlrW_Avg,ETos,Rso"
# classifier = SoilOrMet(current_met_header=MET_HEADER, current_soil_header=SOIL_HEADER,
#                        min_soil_features=9, min_met_features=10)

classifier = SoilOrMet()

print(f"Met  file classified as: {classifier.determine_data_file(example_met_file)}")
print(f"Soil file classified as: {classifier.determine_data_file(example_soil_file)}")

Met  file classified as: met
Soil file classified as: soil


---
## 2. `read_data.py`  ·  *library module, no CLI*

`file_to_indexed_df(file_path, is_soil_or_met)` is the entry point. Internally it calls:
- `read_soil()` — skips 5 header lines, uses `Date` as the index, keeps `Flag` as a string
- `read_met()` — skips 6 header lines plus the 2 units/type rows, drops `RECORD`, standardises column names, uses `TIMESTAMP` (renamed `Date`) as the index
- `check_for_NaT()` — warns if any timestamps could not be parsed

Both readers parse the measurement columns as native numerics (coercing junk to `NaN`) and sort the index stably.

In [5]:
from read_data import file_to_indexed_df

soil_raw = file_to_indexed_df(example_soil_file, is_soil_or_met="soil")
print(f"Soil — shape: {soil_raw.shape}, index type: {type(soil_raw.index).__name__}")
soil_raw.head()

Soil — shape: (85252, 10), index type: DatetimeIndex


,Ppt,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag
Date,,,,,,,,,,
2014-09-29 18:00:00,7.87,0.080,0.113,0.072,0.049,36.01,33.13,30.73,27.95,0
2014-09-29 19:00:00,0.00,0.079,0.113,0.073,0.049,34.23,33.07,30.92,27.93,0
2014-09-29 20:00:00,0.00,0.078,0.112,0.073,0.049,31.69,32.50,30.92,27.84,0
2014-09-29 21:00:00,0.00,0.077,0.111,0.073,0.049,29.41,31.69,30.79,27.79,0
2014-09-29 22:00:00,0.00,0.076,0.110,0.072,0.049,27.72,30.79,30.52,27.75,0


In [6]:
met_raw = file_to_indexed_df(example_met_file, is_soil_or_met="met")
print(f"Met  — shape: {met_raw.shape}, index type: {type(met_raw.index).__name__}")
met_raw.head()

Met  — shape: (15598, 8), index type: DatetimeIndex


,Ppt,Tair,RH,Wind speed,Wind direction,Srad,ETos,Rso
Date,,,,,,,,
2022-06-08 12:00:00,0.0,33.16,36.49,1.191,168.20,990.0,0.503,3.594
2022-06-08 13:00:00,0.0,34.30,29.99,0.745,189.30,1014.0,0.800,3.503
2022-06-08 14:00:00,0.0,35.98,28.25,0.677,126.60,981.0,0.787,3.204
2022-06-08 15:00:00,0.0,37.27,28.61,1.997,90.30,911.0,0.804,2.699
2022-06-08 16:00:00,0.0,36.35,30.99,3.137,77.67,757.6,0.727,2.034


---
## 3. `dup_cleaner.py`

`dup_cleaner(df)` resolves duplicate timestamps in three passes (all keep the first occurrence):

| Case | Condition |
|---|---|
| 1 | Same timestamp + same measurements + same flag |
| 2 | Same timestamp + same measurements + different flag |
| 3 | Same timestamp + different measurements |

Requires a `DatetimeIndex`. Returns a deduplicated DataFrame with the `DatetimeIndex` restored.

**CLI**
```bash
python dup_cleaner.py <input.dat> <output.csv>
```

In [7]:
from dup_cleaner import dup_cleaner

n_before = soil_raw.index.duplicated().sum()
soil_deduped = dup_cleaner(soil_raw)
n_after = soil_deduped.index.duplicated().sum()

print(f"Soil — duplicate timestamps before: {n_before}, after: {n_after}")
print(f"Rows removed: {len(soil_raw) - len(soil_deduped)}")

Soil — duplicate timestamps before: 4077, after: 0
Rows removed: 4077


---
## 4. `treat_subhourly_data.py`

`treat_subhourly_data(df)` condenses sub-hourly readings (timestamps with non-zero minutes) onto the hourly grid:
- sub-hourly **precipitation** (`Ppt`) is summed into the next whole hour, so no rainfall is lost
- every other measurement keeps the value already recorded on the whole hour; the sub-hourly rows are dropped

Runs **after** `dup_cleaner` and **before** `time_cleaner`. In this dataset only **FD08** has many sub-hourly timestamps, so we use it to show the effect.

**CLI**
```bash
python treat_subhourly_data.py <input.dat> <output.csv>
```

In [8]:
from treat_subhourly_data import treat_subhourly_data

# keep the running example moving through the pipeline (dup -> sub-hourly -> time ...)
soil_condensed = treat_subhourly_data(soil_deduped)

# FD08 is the station that actually has many sub-hourly timestamps — use it to show the effect
fd08 = dup_cleaner(file_to_indexed_df(os.path.join(DATA_FOLDER, "FD08.dat"), "soil"))
n_before = int((fd08.index != fd08.index.floor("h")).sum())
fd08_condensed = treat_subhourly_data(fd08)
n_after = int((fd08_condensed.index != fd08_condensed.index.floor("h")).sum())

print(f"FD08 — sub-hourly timestamps before: {n_before}, after: {n_after}")
print(f"FD08 — rows: {len(fd08)} -> {len(fd08_condensed)}")

FD08 — sub-hourly timestamps before: 24716, after: 0
FD08 — rows: 106360 -> 81857


---
## 5. `time_cleaner.py`

`time_cleaner(df)` fills gaps in an hourly `DatetimeIndex` by reindexing over the full `date_range` from the first to last timestamp. Missing rows are inserted with `NaN` measurements.

Must run **after** `dup_cleaner` and `treat_subhourly_data` (it needs a unique, whole-hour `DatetimeIndex`).

**CLI**
```bash
python time_cleaner.py <input.dat> <output.csv>
```

In [9]:
from time_cleaner import time_cleaner

soil_clean = time_cleaner(soil_condensed)

print(f"Soil — rows before: {len(soil_condensed)}, after: {len(soil_clean)}")
print(f"Timestamps filled in with NaN: {len(soil_clean) - len(soil_condensed)}")

Soil — rows before: 81175, after: 83007
Timestamps filled in with NaN: 1832


---
## 6. `treat_wrong_data.py`

`find_and_replace_wrong_data(df)` replaces out-of-range measurement values with `NA`: soil moisture outside 0–0.6, `RH` outside 0–100, wind speed outside 0–25, wind direction outside 0–360, `Srad`/`Ppt` below 0, and temperatures outside −30–60 °C. The `Flag` column is left untouched.

**CLI**
```bash
python treat_wrong_data.py <input.dat> <output.csv>
```

In [10]:
from treat_wrong_data import find_and_replace_wrong_data

print("Out-of-range values become NA. NA count per column before:")
print(soil_clean.isna().sum())

soil_treated = find_and_replace_wrong_data(soil_clean)

print("\nNA count per column after:")
print(soil_treated.isna().sum())

Out-of-range values become NA. NA count per column before:
Ppt       1832
SWC_5     1832
SWC_10    1832
SWC_20    1832
SWC_50    1832
T_5       1832
T_10      1832
T_20      1832
T_50      1832
Flag      1832
dtype: int64

NA count per column after:
Ppt       1832
SWC_5     1895
SWC_10    1939
SWC_20    2244
SWC_50    1857
T_5       1832
T_10      1832
T_20      1832
T_50      1832
Flag      1832
dtype: int64


---
## 7. `gap_report.py`

`gap_report(df)` tallies gaps by duration (`<24h`, `1-7d`, `7-30d`, `>30d`) for both the timestamp index (missing hours) and each column (consecutive-`NaN` runs), printing and returning the table. Run it on the deduped data to see the gaps `time_cleaner` will fill.

**CLI** — the output CSV is optional
```bash
python gap_report.py <input.dat> [output.csv]
```

In [19]:
from gap_report import gap_report

report_before = gap_report(soil_deduped)
report_after = gap_report(soil_treated)

            <24h  1-7d  7-30d  >30d
gaps_in                            
timestamps     2     0      1     1
Ppt            0     0      0     0
SWC_5          0     0      0     0
SWC_10         0     0      0     0
SWC_20         0     0      0     0
SWC_50         0     0      0     0
T_5            0     0      0     0
T_10           0     0      0     0
T_20           0     0      0     0
T_50           0     0      0     0
Flag           0     0      0     0
            <24h  1-7d  7-30d  >30d
gaps_in                            
timestamps     0     0      0     0
Ppt            2     0      1     1
SWC_5          7     0      1     1
SWC_10        14     0      1     1
SWC_20        24     3      1     1
SWC_50         2     1      1     1
T_5            2     0      1     1
T_10           2     0      1     1
T_20           2     0      1     1
T_50           2     0      1     1
Flag           2     0      1     1


---
## 8. `get_data_dict.py`

`data_ingest` runs the full pipeline over every `.dat` file in a folder:

1. `open_df()` — classifies each file with `SoilOrMet` and reads it with `read_data`
2. `prewash_data(df)` — for one station, applies `dup_cleaner` → `treat_subhourly_data` → `time_cleaner` → `find_and_replace_wrong_data` (when `prewash_the_data=True`)
3. `clean_data()` — *not yet implemented* (will impute values)
4. `get_data_dict()` — orchestrates the above and returns `(met_dict, soil_dict)`; with `download=True` each prewashed station is also written to `prewash_folder`

Dict keys are station names from the filename (e.g. `"CB01_met"`, `"CB01"`).

**CLI**
```bash
python get_data_dict.py <input_folder> --prewash --download [--prewash-folder DIR]
```

In [ ]:
from get_data_dict import data_ingest

ingest = data_ingest(
    input_data_folder=DATA_FOLDER,
    prewash_the_data=True,
    # download=True, # adds significant time (~ 12 seconds)
    # prewash_folder=None,  # None -> "prewashed_data" in the working directory
)

met_dict, soil_dict = ingest.get_data_dict()

print(f"Met  stations loaded : {len(met_dict)}  — {sorted(met_dict)}")
print(f"Soil stations loaded : {len(soil_dict)} — {sorted(soil_dict)}")

Met  stations loaded : 6  — ['CB01_met', 'CB04_met', 'CB06_met', 'FD02_met', 'FD03_met', 'WC05_met']
Soil stations loaded : 33 — ['CB01', 'CB04', 'CB06', 'CB07', 'CB09', 'CB10', 'CB15', 'CB19', 'CB20', 'CB25', 'CB26', 'CB27', 'CB31', 'CB32', 'CB33', 'FD02', 'FD03', 'FD08', 'FD11', 'FD12', 'FD13', 'FD14', 'FD16', 'FD17', 'FD18', 'FD21', 'FD22', 'FD23', 'FD24', 'FD28', 'FD29', 'FD30', 'WC05']


In [17]:
# Inspect one station from each dict
first_met_key  = next(iter(met_dict))
first_soil_key = next(iter(soil_dict))

print(f"=== Met  [{first_met_key}] — {met_dict[first_met_key].shape} ===")
display(met_dict[first_met_key].head())

print(f"=== Soil [{first_soil_key}] — {soil_dict[first_soil_key].shape} ===")
display(soil_dict[first_soil_key].head())

=== Met  [CB01_met] — (15597, 8) ===


,Ppt,Tair,RH,Wind speed,Wind direction,Srad,ETos,Rso
2022-06-08 12:00:00,0.0,33.16,36.49,1.191,168.20,990.0,0.503,3.594
2022-06-08 13:00:00,0.0,34.30,29.99,0.745,189.30,1014.0,0.800,3.503
2022-06-08 14:00:00,0.0,35.98,28.25,0.677,126.60,981.0,0.787,3.204
2022-06-08 15:00:00,0.0,37.27,28.61,1.997,90.30,911.0,0.804,2.699
2022-06-08 16:00:00,0.0,36.35,30.99,3.137,77.67,757.6,0.727,2.034


=== Soil [CB01] — (83007, 10) ===


,Ppt,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag
2014-09-29 18:00:00,7.87,0.080,0.113,0.072,0.049,36.01,33.13,30.73,27.95,0
2014-09-29 19:00:00,0.00,0.079,0.113,0.073,0.049,34.23,33.07,30.92,27.93,0
2014-09-29 20:00:00,0.00,0.078,0.112,0.073,0.049,31.69,32.50,30.92,27.84,0
2014-09-29 21:00:00,0.00,0.077,0.111,0.073,0.049,29.41,31.69,30.79,27.79,0
2014-09-29 22:00:00,0.00,0.076,0.110,0.072,0.049,27.72,30.79,30.52,27.75,0
